In [221]:
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

In [222]:
fecha = '20260414'

Paraderos

In [223]:
#importar paradas

paradas = pd.read_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Paraderos_Zonales_del_SITP.csv')

paradas.head(3)

,X,Y,objectid,cenefa,zona_sitp,nombre,via,direccion_bandera,localidad,longitud,latitud,consecutivo_zona,tipo_m_s,consola,panel,audio,zonas_nuevas,globalid,shape
0,1.001502e+06,1.010205e+06,1,001A00,00,C.C. Iserra 100,AC 100,AC 100 - KR 54,Barrios Unidos,-74.063971,4.688481,001,M,AC 100 - KR 54 (001A00),AC 100 - KR 54,Avenida Calle 100 Carrera 54,C,{1C0DBC4E-15BC-4BBE-BC16-5628077DBE2E},NaN
1,1.003505e+06,1.009719e+06,2,001A01,01,Br. Rincón del Chicó,AC 100,AC 100 - KR 13,Usaquén,-74.045914,4.684091,001,M,AC 100 - KR 13 (001A01),AC 100 - KR 13,Avenida Calle 100 Carrera 13,B,{60A22A44-AD56-4DF4-A3E6-6830B9FB0792},NaN
2,1.001238e+06,1.018098e+06,3,001A02,02,Gimnasio Iragua,AV. Boyacá,AV. Boyacá - AC 170,Suba,-74.066350,4.759867,001,S,AV. Boyacá - AC 170 (001A02),AV. Boyacá - AC 170,Avenida Boyacá Avenida Calle 170,C,{97155274-E4D5-45A4-891F-2DCA19FF10D9},NaN


Matriz de distancia

In [224]:
#Zonal

md_zonal = pd.read_csv(f'Z:/01 base_datos/06 matriz_distancia_FMS/{fecha}_matriz distancias.csv', encoding='latin')

#Troncal

md_troncal = pd.read_csv(f'Z:/01 base_datos/31 matriz_distancia_troncal_FMS/{fecha}_matriz_distancias_troncal.csv', encoding='latin')

md = pd.concat([md_zonal, md_troncal])

md.head(2)

,ï»¿Tipo de Servicio,Id LÃ­nea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,PosiciÃ³n,Coordenada X,Coordenada Y,Atributos
0,URBANO,10184,740,1,2.0,338.0,10415.0,740_V1,Circular,52845.0,247A05_TM,247A05_Br. La Esperanza II,0.0,595348.0,521193.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,URBANO,10184,740,1,2.0,338.0,10415.0,740_V1,Circular,52806.0,222A05_TM,222A05_Hospital de EngativÃ¡ EmaÃºs,138.0,595258.0,521149.0,NaN


In [225]:
md = md[
    md["Id Nodo"].notna() &
    (md["Id Nodo"].astype(str).str.strip() != "") &
    (~md["Id Nodo"].astype(str).str.lower().isin(["nan"]))
]

md["Id Nodo"] = md["Id Nodo"].astype(int)
md["Id Ruta"] = md["Id Ruta"].astype(int)

md.head()

,ï»¿Tipo de Servicio,Id LÃ­nea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,PosiciÃ³n,Coordenada X,Coordenada Y,Atributos
0,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52845,247A05_TM,247A05_Br. La Esperanza II,0.0,595348.0,521193.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52806,222A05_TM,222A05_Hospital de EngativÃ¡ EmaÃºs,138.0,595258.0,521149.0,NaN
2,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52365,068A05_TM,068A05_Liceo SalomÃ³n Sabio,425.0,595014.0,520997.0,NaN
3,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52442,110A05_TM,110A05_Br. Sabana del Dorado,686.0,594933.0,520819.0,NaN
4,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52369,070A05_TM,070A05_Br. Sabana del Dorado,964.0,595091.0,520593.0,NaN


In [226]:
#Cruzar datos con vd y md

md = md.rename(columns={
    "Id LÃ­nea": "Id Línea",
    "PosiciÃ³n": "Posición"
})

md.head(3)

,ï»¿Tipo de Servicio,Id Línea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posición,Coordenada X,Coordenada Y,Atributos
0,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52845,247A05_TM,247A05_Br. La Esperanza II,0.0,595348.0,521193.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52806,222A05_TM,222A05_Hospital de EngativÃ¡ EmaÃºs,138.0,595258.0,521149.0,NaN
2,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52365,068A05_TM,068A05_Liceo SalomÃ³n Sabio,425.0,595014.0,520997.0,NaN


In [227]:
md = md.sort_values(
    by=['Id Línea', 'Id Ruta', 'Posición']
).reset_index(drop=True)

md.head(2)

,ï»¿Tipo de Servicio,Id Línea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posición,Coordenada X,Coordenada Y,Atributos
0,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71475,193B03,Av Suba - K114D,0.0,599402.0,525137.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71476,195B03,Av Suba - K110A.,453.0,599813.0,524944.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."


In [228]:
md['orden'] = md.groupby(
    ['Id Línea', 'Id Ruta']
).cumcount() + 1

md.head(2)

,ï»¿Tipo de Servicio,Id Línea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posición,Coordenada X,Coordenada Y,Atributos,orden
0,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71475,193B03,Av Suba - K114D,0.0,599402.0,525137.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM...",1
1,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71476,195B03,Av Suba - K110A.,453.0,599813.0,524944.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM...",2


In [229]:
md["Parada"] = md["Etiqueta Nodo"].str.split("_").str[0]

md.head()

,ï»¿Tipo de Servicio,Id Línea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posición,Coordenada X,Coordenada Y,Atributos,orden,Parada
0,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71475,193B03,Av Suba - K114D,0.0,599402.0,525137.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM...",1,193B03
1,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71476,195B03,Av Suba - K110A.,453.0,599813.0,524944.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM...",2,195B03
2,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,61413,C004,21 Ãngeles A - 2,3097.0,601949.0,523468.0,"SinÃ³ptico, Punto de control (TM-GOAL)",3,C004
3,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,61441,C032,Puentelargo A - 2,8326.0,603467.0,518807.0,SinÃ³ptico,4,C032
4,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,61523,D054,Polo C - 2 Ã³ 5,11858.0,603726.0,516325.0,"SinÃ³ptico, Punto de control (TM-GOAL)",5,D054


Validaciones

In [230]:
import os
import pandas as pd

ruta = f"Z:/01 base_datos/41 Informe Diario CCZ/FMS 2025/Validaciones_diarias/{fecha}_validacionZonal.csv"

validaciones = pd.read_csv(ruta, low_memory=False)

# Extrae el nombre del archivo automáticamente
validaciones["Fecha_archivo"] = os.path.basename(ruta).split("_")[0]

validaciones = validaciones[validaciones['Operador'] == '(005) GMOVIL ENGATIVA']

validaciones.head(3)

,Dispositivo,Emisor,Estacion_Parada,Fase,Fecha_Clearing,Fecha_Transaccion,Hora_Pico_SN,ID_Vehiculo,Linea,Nombre_Perfil,...,Operador,Ruta,Saldo_Despues_Transaccion,Saldo_Previo_a_Transaccion,Sistema,Tipo_Tarifa,Tipo_Tarjeta,Tipo_Vehiculo,Valor,Fecha_archivo
1679669,220005008,(3101000) Bogota Card(Citizen),(54101) 473A12_TM|473A12_Br. Gran Yomasa II Se...,Fase 3,2026-04-14,2026-04-14 03:56:59,Peak Time,504284,(10264) 614,(002) Adulto Mayor,...,(005) GMOVIL ENGATIVA,(12328) 614_Vuelta_V2,15340.0,18890.0,ZONAL,1,tullave Plus,(02) Urbano,3550.0,20260414
1679670,220009128,(3101000) Bogota Card(Citizen),(52765) 204B05_TM|204B05_Br. San Antonio Engativá,Fase 3,2026-04-14,2026-04-14 03:57:08,Peak Time,504495,(10184) 740,(001) Adulto,...,(005) GMOVIL ENGATIVA,(12774) 740_V2,8300.0,11850.0,ZONAL,1,tullave Plus,(02) Urbano,3550.0,20260414
1679671,220001856,(3101000) Bogota Card(Citizen),(54051) 438A12_TM|438A12_Br. Los Molinos II Se...,Fase 3,2026-04-14,2026-04-14 03:57:28,Peak Time,504163,(10264) 614,(001) Adulto,...,(005) GMOVIL ENGATIVA,(12328) 614_Vuelta_V2,32250.0,35800.0,ZONAL,1,tullave Plus,(02) Urbano,3550.0,20260414


In [231]:
validaciones = validaciones.drop([
    'Dispositivo', 'Emisor','Fase', 'Hora_Pico_SN','Nombre_Perfil', 
    'Operador', 'Saldo_Despues_Transaccion', 
    'Saldo_Previo_a_Transaccion', 'Sistema', 
    'Tipo_Tarifa', 'Tipo_Tarjeta','Valor'
], axis=1)

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo
1679669,(54101) 473A12_TM|473A12_Br. Gran Yomasa II Se...,2026-04-14,2026-04-14 03:56:59,504284,(10264) 614,2a9cd322b12f8e3d4f0ddd035b87638e561d5ebddb7af0...,(12328) 614_Vuelta_V2,(02) Urbano,20260414
1679670,(52765) 204B05_TM|204B05_Br. San Antonio Engativá,2026-04-14,2026-04-14 03:57:08,504495,(10184) 740,345b3e530957e2d8736ea90c5160fd134d6642d17f8697...,(12774) 740_V2,(02) Urbano,20260414
1679671,(54051) 438A12_TM|438A12_Br. Los Molinos II Se...,2026-04-14,2026-04-14 03:57:28,504163,(10264) 614,b4a06d871b84ca5ed8fc9ff9e483e159e099e8ed9dff45...,(12328) 614_Vuelta_V2,(02) Urbano,20260414
1679672,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:49,504177,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260414
1679673,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:54,504177,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260414


In [232]:
# Convertir la columna 'Fecha_Transaccion' a tipo datetime
validaciones['Fecha_Transaccion'] = pd.to_datetime(validaciones['Fecha_Transaccion'])

# Extraer la parte de la hora y colocarla en una nueva columna 'Hora_Transaccion'
validaciones['Hora_Transaccion'] = validaciones['Fecha_Transaccion'].dt.time

# Mostrar el DataFrame resultante
validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion
1679669,(54101) 473A12_TM|473A12_Br. Gran Yomasa II Se...,2026-04-14,2026-04-14 03:56:59,504284,(10264) 614,2a9cd322b12f8e3d4f0ddd035b87638e561d5ebddb7af0...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:56:59
1679670,(52765) 204B05_TM|204B05_Br. San Antonio Engativá,2026-04-14,2026-04-14 03:57:08,504495,(10184) 740,345b3e530957e2d8736ea90c5160fd134d6642d17f8697...,(12774) 740_V2,(02) Urbano,20260414,03:57:08
1679671,(54051) 438A12_TM|438A12_Br. Los Molinos II Se...,2026-04-14,2026-04-14 03:57:28,504163,(10264) 614,b4a06d871b84ca5ed8fc9ff9e483e159e099e8ed9dff45...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:28
1679672,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:49,504177,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:49
1679673,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:54,504177,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:54


In [233]:
# Utilizando expresiones regulares para extraer el número entre paréntesis y el resto del texto
validaciones[['Numero_parada', 'Nombre_Completo']] = validaciones['Estacion_Parada'].str.extract(r'\((.*?)\)(.*)')

# Dividir la columna "Resto" en dos partes usando el carácter '|' como separador
validaciones[['Parada', 'Nombre']] = validaciones['Nombre_Completo'].str.split('|', expand=True)

# Mostrar el DataFrame resultante
validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,Numero_parada,Nombre_Completo,Parada,Nombre
1679669,(54101) 473A12_TM|473A12_Br. Gran Yomasa II Se...,2026-04-14,2026-04-14 03:56:59,504284,(10264) 614,2a9cd322b12f8e3d4f0ddd035b87638e561d5ebddb7af0...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:56:59,54101,473A12_TM|473A12_Br. Gran Yomasa II Sector,473A12_TM,473A12_Br. Gran Yomasa II Sector
1679670,(52765) 204B05_TM|204B05_Br. San Antonio Engativá,2026-04-14,2026-04-14 03:57:08,504495,(10184) 740,345b3e530957e2d8736ea90c5160fd134d6642d17f8697...,(12774) 740_V2,(02) Urbano,20260414,03:57:08,52765,204B05_TM|204B05_Br. San Antonio Engativá,204B05_TM,204B05_Br. San Antonio Engativá
1679671,(54051) 438A12_TM|438A12_Br. Los Molinos II Se...,2026-04-14,2026-04-14 03:57:28,504163,(10264) 614,b4a06d871b84ca5ed8fc9ff9e483e159e099e8ed9dff45...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:28,54051,438A12_TM|438A12_Br. Los Molinos II Sector,438A12_TM,438A12_Br. Los Molinos II Sector
1679672,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:49,504177,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:49,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur
1679673,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:54,504177,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:54,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur


In [234]:
# Utilizando expresiones regulares para extraer el número entre paréntesis y el resto del texto
validaciones[['Linea_1', 'Ruta_comercial']] = validaciones['Linea'].str.extract(r'\((.*?)\)(.*)')

validaciones[['Ruta_SAE', 'Sentido']] = validaciones['Ruta'].str.extract(r'\((.*?)\)(.*)')

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,Numero_parada,Nombre_Completo,Parada,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido
1679669,(54101) 473A12_TM|473A12_Br. Gran Yomasa II Se...,2026-04-14,2026-04-14 03:56:59,504284,(10264) 614,2a9cd322b12f8e3d4f0ddd035b87638e561d5ebddb7af0...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:56:59,54101,473A12_TM|473A12_Br. Gran Yomasa II Sector,473A12_TM,473A12_Br. Gran Yomasa II Sector,10264,614,12328,614_Vuelta_V2
1679670,(52765) 204B05_TM|204B05_Br. San Antonio Engativá,2026-04-14,2026-04-14 03:57:08,504495,(10184) 740,345b3e530957e2d8736ea90c5160fd134d6642d17f8697...,(12774) 740_V2,(02) Urbano,20260414,03:57:08,52765,204B05_TM|204B05_Br. San Antonio Engativá,204B05_TM,204B05_Br. San Antonio Engativá,10184,740,12774,740_V2
1679671,(54051) 438A12_TM|438A12_Br. Los Molinos II Se...,2026-04-14,2026-04-14 03:57:28,504163,(10264) 614,b4a06d871b84ca5ed8fc9ff9e483e159e099e8ed9dff45...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:28,54051,438A12_TM|438A12_Br. Los Molinos II Sector,438A12_TM,438A12_Br. Los Molinos II Sector,10264,614,12328,614_Vuelta_V2
1679672,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:49,504177,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:49,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2
1679673,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:54,504177,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:54,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2


In [235]:
validaciones["Hora_Transaccion"] = validaciones["Hora_Transaccion"].astype(str).str.strip()

# Convertir "nan" y vacíos a NaN reales
validaciones.loc[
    validaciones["Hora_Transaccion"].isin(["", "nan", "None"]),
    "Hora_Transaccion"
] = None

In [236]:
validaciones["Franja"] = pd.to_datetime(
    validaciones["Hora_Transaccion"], errors="coerce"
).dt.hour

validaciones.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_28816\251133772.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  validaciones["Franja"] = pd.to_datetime(


,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,Numero_parada,Nombre_Completo,Parada,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja
1679669,(54101) 473A12_TM|473A12_Br. Gran Yomasa II Se...,2026-04-14,2026-04-14 03:56:59,504284,(10264) 614,2a9cd322b12f8e3d4f0ddd035b87638e561d5ebddb7af0...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:56:59,54101,473A12_TM|473A12_Br. Gran Yomasa II Sector,473A12_TM,473A12_Br. Gran Yomasa II Sector,10264,614,12328,614_Vuelta_V2,3
1679670,(52765) 204B05_TM|204B05_Br. San Antonio Engativá,2026-04-14,2026-04-14 03:57:08,504495,(10184) 740,345b3e530957e2d8736ea90c5160fd134d6642d17f8697...,(12774) 740_V2,(02) Urbano,20260414,03:57:08,52765,204B05_TM|204B05_Br. San Antonio Engativá,204B05_TM,204B05_Br. San Antonio Engativá,10184,740,12774,740_V2,3
1679671,(54051) 438A12_TM|438A12_Br. Los Molinos II Se...,2026-04-14,2026-04-14 03:57:28,504163,(10264) 614,b4a06d871b84ca5ed8fc9ff9e483e159e099e8ed9dff45...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:28,54051,438A12_TM|438A12_Br. Los Molinos II Sector,438A12_TM,438A12_Br. Los Molinos II Sector,10264,614,12328,614_Vuelta_V2,3
1679672,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:49,504177,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:49,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3
1679673,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:54,504177,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:54,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3


In [237]:
validaciones["id_tarjeta"] = validaciones["Numero_Tarjeta"].astype("category").cat.codes

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,Numero_parada,Nombre_Completo,Parada,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,id_tarjeta
1679669,(54101) 473A12_TM|473A12_Br. Gran Yomasa II Se...,2026-04-14,2026-04-14 03:56:59,504284,(10264) 614,2a9cd322b12f8e3d4f0ddd035b87638e561d5ebddb7af0...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:56:59,54101,473A12_TM|473A12_Br. Gran Yomasa II Sector,473A12_TM,473A12_Br. Gran Yomasa II Sector,10264,614,12328,614_Vuelta_V2,3,21892
1679670,(52765) 204B05_TM|204B05_Br. San Antonio Engativá,2026-04-14,2026-04-14 03:57:08,504495,(10184) 740,345b3e530957e2d8736ea90c5160fd134d6642d17f8697...,(12774) 740_V2,(02) Urbano,20260414,03:57:08,52765,204B05_TM|204B05_Br. San Antonio Engativá,204B05_TM,204B05_Br. San Antonio Engativá,10184,740,12774,740_V2,3,27072
1679671,(54051) 438A12_TM|438A12_Br. Los Molinos II Se...,2026-04-14,2026-04-14 03:57:28,504163,(10264) 614,b4a06d871b84ca5ed8fc9ff9e483e159e099e8ed9dff45...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:28,54051,438A12_TM|438A12_Br. Los Molinos II Sector,438A12_TM,438A12_Br. Los Molinos II Sector,10264,614,12328,614_Vuelta_V2,3,94261
1679672,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:49,504177,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:49,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,38898
1679673,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:54,504177,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:54,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,132557


In [238]:
validaciones["Parada"] = validaciones["Parada"].str.split("_").str[0]

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,Numero_parada,Nombre_Completo,Parada,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,id_tarjeta
1679669,(54101) 473A12_TM|473A12_Br. Gran Yomasa II Se...,2026-04-14,2026-04-14 03:56:59,504284,(10264) 614,2a9cd322b12f8e3d4f0ddd035b87638e561d5ebddb7af0...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:56:59,54101,473A12_TM|473A12_Br. Gran Yomasa II Sector,473A12,473A12_Br. Gran Yomasa II Sector,10264,614,12328,614_Vuelta_V2,3,21892
1679670,(52765) 204B05_TM|204B05_Br. San Antonio Engativá,2026-04-14,2026-04-14 03:57:08,504495,(10184) 740,345b3e530957e2d8736ea90c5160fd134d6642d17f8697...,(12774) 740_V2,(02) Urbano,20260414,03:57:08,52765,204B05_TM|204B05_Br. San Antonio Engativá,204B05,204B05_Br. San Antonio Engativá,10184,740,12774,740_V2,3,27072
1679671,(54051) 438A12_TM|438A12_Br. Los Molinos II Se...,2026-04-14,2026-04-14 03:57:28,504163,(10264) 614,b4a06d871b84ca5ed8fc9ff9e483e159e099e8ed9dff45...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:28,54051,438A12_TM|438A12_Br. Los Molinos II Sector,438A12,438A12_Br. Los Molinos II Sector,10264,614,12328,614_Vuelta_V2,3,94261
1679672,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:49,504177,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:49,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,38898
1679673,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:54,504177,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:54,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,132557


In [239]:
#convertir a entero datos de linea y ruta

validaciones["Linea_1"] = validaciones["Linea_1"].astype(int)
validaciones["Ruta_SAE"] = validaciones["Ruta_SAE"].astype(int)
validaciones["Numero_parada"] = validaciones["Numero_parada"].astype(int)

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,Numero_parada,Nombre_Completo,Parada,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,id_tarjeta
1679669,(54101) 473A12_TM|473A12_Br. Gran Yomasa II Se...,2026-04-14,2026-04-14 03:56:59,504284,(10264) 614,2a9cd322b12f8e3d4f0ddd035b87638e561d5ebddb7af0...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:56:59,54101,473A12_TM|473A12_Br. Gran Yomasa II Sector,473A12,473A12_Br. Gran Yomasa II Sector,10264,614,12328,614_Vuelta_V2,3,21892
1679670,(52765) 204B05_TM|204B05_Br. San Antonio Engativá,2026-04-14,2026-04-14 03:57:08,504495,(10184) 740,345b3e530957e2d8736ea90c5160fd134d6642d17f8697...,(12774) 740_V2,(02) Urbano,20260414,03:57:08,52765,204B05_TM|204B05_Br. San Antonio Engativá,204B05,204B05_Br. San Antonio Engativá,10184,740,12774,740_V2,3,27072
1679671,(54051) 438A12_TM|438A12_Br. Los Molinos II Se...,2026-04-14,2026-04-14 03:57:28,504163,(10264) 614,b4a06d871b84ca5ed8fc9ff9e483e159e099e8ed9dff45...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:28,54051,438A12_TM|438A12_Br. Los Molinos II Sector,438A12,438A12_Br. Los Molinos II Sector,10264,614,12328,614_Vuelta_V2,3,94261
1679672,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:49,504177,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:49,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,38898
1679673,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:54,504177,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:54,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,132557


In [240]:
validaciones["Parada"] = validaciones["Parada"].astype(str).str.strip().str.upper()

validaciones = validaciones[validaciones["Parada"] != "(UNKNOWN)"]

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,Numero_parada,Nombre_Completo,Parada,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,id_tarjeta
1679669,(54101) 473A12_TM|473A12_Br. Gran Yomasa II Se...,2026-04-14,2026-04-14 03:56:59,504284,(10264) 614,2a9cd322b12f8e3d4f0ddd035b87638e561d5ebddb7af0...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:56:59,54101,473A12_TM|473A12_Br. Gran Yomasa II Sector,473A12,473A12_Br. Gran Yomasa II Sector,10264,614,12328,614_Vuelta_V2,3,21892
1679670,(52765) 204B05_TM|204B05_Br. San Antonio Engativá,2026-04-14,2026-04-14 03:57:08,504495,(10184) 740,345b3e530957e2d8736ea90c5160fd134d6642d17f8697...,(12774) 740_V2,(02) Urbano,20260414,03:57:08,52765,204B05_TM|204B05_Br. San Antonio Engativá,204B05,204B05_Br. San Antonio Engativá,10184,740,12774,740_V2,3,27072
1679671,(54051) 438A12_TM|438A12_Br. Los Molinos II Se...,2026-04-14,2026-04-14 03:57:28,504163,(10264) 614,b4a06d871b84ca5ed8fc9ff9e483e159e099e8ed9dff45...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:28,54051,438A12_TM|438A12_Br. Los Molinos II Sector,438A12,438A12_Br. Los Molinos II Sector,10264,614,12328,614_Vuelta_V2,3,94261
1679672,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:49,504177,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:49,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,38898
1679673,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:54,504177,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:54,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,132557


In [241]:
#Cruzar con datos de nodo y Numero_parada

def calcular_turno(linea, ruta, nodo):
    
    filtro = (
        (md['Id Línea'] == linea)&
        (md['Id Ruta'] == ruta)&
        (md['Id Nodo'] == nodo)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not md.loc[filtro].empty:
        # Obtener el primer valor
        mds = md.loc[filtro, 'orden'].iloc[0]
        return mds if not pd.isna(mds) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
validaciones['orden_parada'] = validaciones.apply(
    lambda row: calcular_turno(
        row['Linea_1'],
        row['Ruta_SAE'],
        row['Numero_parada']
    ),
    axis=1
)

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,...,Nombre_Completo,Parada,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,id_tarjeta,orden_parada
1679669,(54101) 473A12_TM|473A12_Br. Gran Yomasa II Se...,2026-04-14,2026-04-14 03:56:59,504284,(10264) 614,2a9cd322b12f8e3d4f0ddd035b87638e561d5ebddb7af0...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:56:59,...,473A12_TM|473A12_Br. Gran Yomasa II Sector,473A12,473A12_Br. Gran Yomasa II Sector,10264,614,12328,614_Vuelta_V2,3,21892,5
1679670,(52765) 204B05_TM|204B05_Br. San Antonio Engativá,2026-04-14,2026-04-14 03:57:08,504495,(10184) 740,345b3e530957e2d8736ea90c5160fd134d6642d17f8697...,(12774) 740_V2,(02) Urbano,20260414,03:57:08,...,204B05_TM|204B05_Br. San Antonio Engativá,204B05,204B05_Br. San Antonio Engativá,10184,740,12774,740_V2,3,27072,10
1679671,(54051) 438A12_TM|438A12_Br. Los Molinos II Se...,2026-04-14,2026-04-14 03:57:28,504163,(10264) 614,b4a06d871b84ca5ed8fc9ff9e483e159e099e8ed9dff45...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:28,...,438A12_TM|438A12_Br. Los Molinos II Sector,438A12,438A12_Br. Los Molinos II Sector,10264,614,12328,614_Vuelta_V2,3,94261,29
1679672,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:49,504177,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:49,...,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,38898,23
1679673,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:54,504177,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:54,...,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,132557,23


In [242]:
paradas["cenefa"] = paradas["cenefa"].astype(str).str.strip().str.upper()


In [243]:
#Cruzar con datos de nodo y Numero_parada

def calcular_turno(parada):
    
    filtro = (
        (paradas['cenefa'] == parada)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not paradas.loc[filtro].empty:
        # Obtener el primer valor
        paradas1 = paradas.loc[filtro, 'latitud'].iloc[0]
        return paradas1 if not pd.isna(paradas1) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
validaciones['latitud'] = validaciones.apply(
    lambda row: calcular_turno(
        row['Parada']
    ),
    axis=1
)

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,...,Parada,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,id_tarjeta,orden_parada,latitud
1679669,(54101) 473A12_TM|473A12_Br. Gran Yomasa II Se...,2026-04-14,2026-04-14 03:56:59,504284,(10264) 614,2a9cd322b12f8e3d4f0ddd035b87638e561d5ebddb7af0...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:56:59,...,473A12,473A12_Br. Gran Yomasa II Sector,10264,614,12328,614_Vuelta_V2,3,21892,5,4.509275
1679670,(52765) 204B05_TM|204B05_Br. San Antonio Engativá,2026-04-14,2026-04-14 03:57:08,504495,(10184) 740,345b3e530957e2d8736ea90c5160fd134d6642d17f8697...,(12774) 740_V2,(02) Urbano,20260414,03:57:08,...,204B05,204B05_Br. San Antonio Engativá,10184,740,12774,740_V2,3,27072,10,4.699254
1679671,(54051) 438A12_TM|438A12_Br. Los Molinos II Se...,2026-04-14,2026-04-14 03:57:28,504163,(10264) 614,b4a06d871b84ca5ed8fc9ff9e483e159e099e8ed9dff45...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:28,...,438A12,438A12_Br. Los Molinos II Sector,10264,614,12328,614_Vuelta_V2,3,94261,29,4.550185
1679672,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:49,504177,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:49,...,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,38898,23,4.542074
1679673,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:54,504177,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:54,...,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,132557,23,4.542074


In [244]:
#Cruzar con datos de nodo y Numero_parada

def calcular_turno(parada):
    
    filtro = (
        (paradas['cenefa'] == parada)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not paradas.loc[filtro].empty:
        # Obtener el primer valor
        paradas1 = paradas.loc[filtro, 'longitud'].iloc[0]
        return paradas1 if not pd.isna(paradas1) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
validaciones['longitud'] = validaciones.apply(
    lambda row: calcular_turno(
        row['Parada']
    ),
    axis=1
)

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,...,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,id_tarjeta,orden_parada,latitud,longitud
1679669,(54101) 473A12_TM|473A12_Br. Gran Yomasa II Se...,2026-04-14,2026-04-14 03:56:59,504284,(10264) 614,2a9cd322b12f8e3d4f0ddd035b87638e561d5ebddb7af0...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:56:59,...,473A12_Br. Gran Yomasa II Sector,10264,614,12328,614_Vuelta_V2,3,21892,5,4.509275,-74.109142
1679670,(52765) 204B05_TM|204B05_Br. San Antonio Engativá,2026-04-14,2026-04-14 03:57:08,504495,(10184) 740,345b3e530957e2d8736ea90c5160fd134d6642d17f8697...,(12774) 740_V2,(02) Urbano,20260414,03:57:08,...,204B05_Br. San Antonio Engativá,10184,740,12774,740_V2,3,27072,10,4.699254,-74.130583
1679671,(54051) 438A12_TM|438A12_Br. Los Molinos II Se...,2026-04-14,2026-04-14 03:57:28,504163,(10264) 614,b4a06d871b84ca5ed8fc9ff9e483e159e099e8ed9dff45...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:28,...,438A12_Br. Los Molinos II Sector,10264,614,12328,614_Vuelta_V2,3,94261,29,4.550185,-74.108529
1679672,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:49,504177,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:49,...,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,38898,23,4.542074,-74.112176
1679673,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-04-14,2026-04-14 03:57:54,504177,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260414,03:57:54,...,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,132557,23,4.542074,-74.112176


Matriz de origen - destino

In [245]:
df_tarjetas = (
    validaciones
    .groupby([
        "Fecha_Clearing",
        "Numero_Tarjeta",
        "id_tarjeta",
        "Numero_parada",
        "Parada",
        "orden_parada",
        "latitud",
        "longitud",
        "Linea_1",
        "Ruta_comercial",
        "Ruta_SAE",
        "Sentido",
        "Franja"
    ])
    .size()
    .reset_index(name="conteo")
)

df_tarjetas.head()

,Fecha_Clearing,Numero_Tarjeta,id_tarjeta,Numero_parada,Parada,orden_parada,latitud,longitud,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,conteo
0,2026-04-14,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,0,52692,181C05,26,4.683280,-74.091076,10688,BD237,12758,BD237_V2,8,1
1,2026-04-14,0000652dcb91a8643aa46f9384dfec0d41bd92a9470e2f...,1,57003,103B00,71,4.638265,-74.069733,10244,SE10,12741,SE10_V3,8,1
2,2026-04-14,00009cecf5dd879db875634471b58705ade02176dde186...,2,51465,157B04,79,4.683434,-74.090934,10688,BD237,12758,BD237_V2,13,1
3,2026-04-14,00009cecf5dd879db875634471b58705ade02176dde186...,2,52370,070B05,5,4.709103,-74.142388,10339,C25,12756,C25_V3,5,1
4,2026-04-14,00009cecf5dd879db875634471b58705ade02176dde186...,2,53008,370A05,106,4.711427,-74.136280,10311,DA213,12730,DA213_V2,13,1


In [246]:
df_tarjetas.groupby(["Numero_Tarjeta", "Franja"])["conteo"].sum()

df_tarjetas.head()

,Fecha_Clearing,Numero_Tarjeta,id_tarjeta,Numero_parada,Parada,orden_parada,latitud,longitud,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,conteo
0,2026-04-14,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,0,52692,181C05,26,4.683280,-74.091076,10688,BD237,12758,BD237_V2,8,1
1,2026-04-14,0000652dcb91a8643aa46f9384dfec0d41bd92a9470e2f...,1,57003,103B00,71,4.638265,-74.069733,10244,SE10,12741,SE10_V3,8,1
2,2026-04-14,00009cecf5dd879db875634471b58705ade02176dde186...,2,51465,157B04,79,4.683434,-74.090934,10688,BD237,12758,BD237_V2,13,1
3,2026-04-14,00009cecf5dd879db875634471b58705ade02176dde186...,2,52370,070B05,5,4.709103,-74.142388,10339,C25,12756,C25_V3,5,1
4,2026-04-14,00009cecf5dd879db875634471b58705ade02176dde186...,2,53008,370A05,106,4.711427,-74.136280,10311,DA213,12730,DA213_V2,13,1


In [247]:
df_tarjetas.groupby(["Numero_Tarjeta", "Parada"])["conteo"].sum()

df_tarjetas.head()

,Fecha_Clearing,Numero_Tarjeta,id_tarjeta,Numero_parada,Parada,orden_parada,latitud,longitud,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,conteo
0,2026-04-14,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,0,52692,181C05,26,4.683280,-74.091076,10688,BD237,12758,BD237_V2,8,1
1,2026-04-14,0000652dcb91a8643aa46f9384dfec0d41bd92a9470e2f...,1,57003,103B00,71,4.638265,-74.069733,10244,SE10,12741,SE10_V3,8,1
2,2026-04-14,00009cecf5dd879db875634471b58705ade02176dde186...,2,51465,157B04,79,4.683434,-74.090934,10688,BD237,12758,BD237_V2,13,1
3,2026-04-14,00009cecf5dd879db875634471b58705ade02176dde186...,2,52370,070B05,5,4.709103,-74.142388,10339,C25,12756,C25_V3,5,1
4,2026-04-14,00009cecf5dd879db875634471b58705ade02176dde186...,2,53008,370A05,106,4.711427,-74.136280,10311,DA213,12730,DA213_V2,13,1


In [248]:
origen = (
    df_tarjetas
    .sort_values(["Numero_Tarjeta", "conteo"], ascending=False)
    .drop_duplicates("Numero_Tarjeta")
)

origen.head()

,Fecha_Clearing,Numero_Tarjeta,id_tarjeta,Numero_parada,Parada,orden_parada,latitud,longitud,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,conteo
166026,2026-04-14,fffff6d484cfcd0551cc32b586251723708e54fe5601d1...,133757,56463,133A01,95,4.760264,-74.027628,10204,402,12776,402_V3,20,1
166024,2026-04-14,fffe518f8fda075b68ab1d9991a7163e835582d549e6cc...,133756,52370,070B05,6,4.709103,-74.142388,10196,E25,12783,E25_V3,5,1
166021,2026-04-14,fffe24146fc9d1f53c0fe56117058e55e9183e5ccb09f3...,133755,51305,104A04,75,4.680931,-74.083067,10688,BD237,12758,BD237_V2,19,1
166020,2026-04-14,fffe23107aecba278a207721e947d438c88c40ce7827d8...,133754,56550,165B01,60,4.684765,-74.036021,10688,BD237,12758,BD237_V2,17,1
166019,2026-04-14,fffd800bf3ae048207f10afe896970e61d97dc676e4e38...,133753,57472,280A00,99,4.654319,-74.058043,10331,12,10632,12,10,1


In [249]:
validaciones["Hora_dt"] = pd.to_datetime(validaciones["Hora_Transaccion"], errors="coerce")

od = (
    validaciones
    .sort_values(["Numero_Tarjeta","id_tarjeta", "Hora_dt"])
    .groupby(["Numero_Tarjeta", "id_tarjeta","Ruta_comercial", "Sentido"])
    .agg(
        origen_parada=("Parada", "first"),
        destino_parada=("Parada", "last"),
        origen_hora=("Hora_dt", "first"),
        destino_hora=("Hora_dt", "last")
    )
    .reset_index()
)

od.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_28816\2080183682.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  validaciones["Hora_dt"] = pd.to_datetime(validaciones["Hora_Transaccion"], errors="coerce")


,Numero_Tarjeta,id_tarjeta,Ruta_comercial,Sentido,origen_parada,destino_parada,origen_hora,destino_hora
0,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,0,BD237,BD237_V2,181C05,181C05,2026-04-16 08:37:14,2026-04-16 08:37:14
1,0000652dcb91a8643aa46f9384dfec0d41bd92a9470e2f...,1,SE10,SE10_V3,103B00,103B00,2026-04-16 08:09:53,2026-04-16 08:09:53
2,00009cecf5dd879db875634471b58705ade02176dde186...,2,BD237,BD237_V2,157B04,157B04,2026-04-16 13:24:44,2026-04-16 13:24:44
3,00009cecf5dd879db875634471b58705ade02176dde186...,2,C25,C25_V3,070B05,070B05,2026-04-16 05:44:54,2026-04-16 05:44:54
4,00009cecf5dd879db875634471b58705ade02176dde186...,2,DA213,DA213_V2,370A05,370A05,2026-04-16 13:51:29,2026-04-16 13:51:29


In [250]:
validaciones = validaciones.sort_values(["id_tarjeta", "Hora_dt"])

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,...,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,id_tarjeta,orden_parada,latitud,longitud,Hora_dt
1734142,(52692) 181C05_TM|181C05_Br. La Estrada,2026-04-14,2026-04-14 08:37:14,502121,(10688) BD237,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,(12758) BD237_V2,(02) Urbano,20260414,08:37:14,...,10688,BD237,12758,BD237_V2,8,0,26,4.683280,-74.091076,2026-04-16 08:37:14
1729151,(57003) 103B00_TM|103B00_Br. Quesada,2026-04-14,2026-04-14 08:09:53,502080,(10244) SE10,0000652dcb91a8643aa46f9384dfec0d41bd92a9470e2f...,(12741) SE10_V3,(02) Urbano,20260414,08:09:53,...,10244,SE10,12741,SE10_V3,8,1,71,4.638265,-74.069733,2026-04-16 08:09:53
1691500,(52370) 070B05_TM|070B05_Br. Sabana del Dorado,2026-04-14,2026-04-14 05:44:54,507061,(10339) C25,00009cecf5dd879db875634471b58705ade02176dde186...,(12756) C25_V3,(02) Urbano,20260414,05:44:54,...,10339,C25,12756,C25_V3,5,2,5,4.709103,-74.142388,2026-04-16 05:44:54
1773353,(51465) 157B04_TM|157B04_Br. Las Ferias,2026-04-14,2026-04-14 13:24:44,502140,(10688) BD237,00009cecf5dd879db875634471b58705ade02176dde186...,(12758) BD237_V2,(02) Urbano,20260414,13:24:44,...,10688,BD237,12758,BD237_V2,13,2,79,4.683434,-74.090934,2026-04-16 13:24:44
1776960,(53008) 370A05_TM|370A05_Br. Nuevo Milenio,2026-04-14,2026-04-14 13:51:29,504389,(10311) DA213,00009cecf5dd879db875634471b58705ade02176dde186...,(12730) DA213_V2,(02) Urbano,20260414,13:51:29,...,10311,DA213,12730,DA213_V2,13,2,106,4.711427,-74.136280,2026-04-16 13:51:29


In [251]:
validaciones["Parada_siguiente"] = validaciones.groupby("id_tarjeta")["Parada"].shift(-1)
validaciones["Hora_siguiente"] = validaciones.groupby("id_tarjeta")["Hora_dt"].shift(-1)

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,...,Ruta_SAE,Sentido,Franja,id_tarjeta,orden_parada,latitud,longitud,Hora_dt,Parada_siguiente,Hora_siguiente
1734142,(52692) 181C05_TM|181C05_Br. La Estrada,2026-04-14,2026-04-14 08:37:14,502121,(10688) BD237,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,(12758) BD237_V2,(02) Urbano,20260414,08:37:14,...,12758,BD237_V2,8,0,26,4.683280,-74.091076,2026-04-16 08:37:14,NaN,NaT
1729151,(57003) 103B00_TM|103B00_Br. Quesada,2026-04-14,2026-04-14 08:09:53,502080,(10244) SE10,0000652dcb91a8643aa46f9384dfec0d41bd92a9470e2f...,(12741) SE10_V3,(02) Urbano,20260414,08:09:53,...,12741,SE10_V3,8,1,71,4.638265,-74.069733,2026-04-16 08:09:53,NaN,NaT
1691500,(52370) 070B05_TM|070B05_Br. Sabana del Dorado,2026-04-14,2026-04-14 05:44:54,507061,(10339) C25,00009cecf5dd879db875634471b58705ade02176dde186...,(12756) C25_V3,(02) Urbano,20260414,05:44:54,...,12756,C25_V3,5,2,5,4.709103,-74.142388,2026-04-16 05:44:54,157B04,2026-04-16 13:24:44
1773353,(51465) 157B04_TM|157B04_Br. Las Ferias,2026-04-14,2026-04-14 13:24:44,502140,(10688) BD237,00009cecf5dd879db875634471b58705ade02176dde186...,(12758) BD237_V2,(02) Urbano,20260414,13:24:44,...,12758,BD237_V2,13,2,79,4.683434,-74.090934,2026-04-16 13:24:44,370A05,2026-04-16 13:51:29
1776960,(53008) 370A05_TM|370A05_Br. Nuevo Milenio,2026-04-14,2026-04-14 13:51:29,504389,(10311) DA213,00009cecf5dd879db875634471b58705ade02176dde186...,(12730) DA213_V2,(02) Urbano,20260414,13:51:29,...,12730,DA213_V2,13,2,106,4.711427,-74.136280,2026-04-16 13:51:29,NaN,NaT


In [252]:
od = validaciones[
    validaciones["Parada"] != validaciones["Parada_siguiente"]
].copy()

od.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,...,Ruta_SAE,Sentido,Franja,id_tarjeta,orden_parada,latitud,longitud,Hora_dt,Parada_siguiente,Hora_siguiente
1734142,(52692) 181C05_TM|181C05_Br. La Estrada,2026-04-14,2026-04-14 08:37:14,502121,(10688) BD237,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,(12758) BD237_V2,(02) Urbano,20260414,08:37:14,...,12758,BD237_V2,8,0,26,4.683280,-74.091076,2026-04-16 08:37:14,NaN,NaT
1729151,(57003) 103B00_TM|103B00_Br. Quesada,2026-04-14,2026-04-14 08:09:53,502080,(10244) SE10,0000652dcb91a8643aa46f9384dfec0d41bd92a9470e2f...,(12741) SE10_V3,(02) Urbano,20260414,08:09:53,...,12741,SE10_V3,8,1,71,4.638265,-74.069733,2026-04-16 08:09:53,NaN,NaT
1691500,(52370) 070B05_TM|070B05_Br. Sabana del Dorado,2026-04-14,2026-04-14 05:44:54,507061,(10339) C25,00009cecf5dd879db875634471b58705ade02176dde186...,(12756) C25_V3,(02) Urbano,20260414,05:44:54,...,12756,C25_V3,5,2,5,4.709103,-74.142388,2026-04-16 05:44:54,157B04,2026-04-16 13:24:44
1773353,(51465) 157B04_TM|157B04_Br. Las Ferias,2026-04-14,2026-04-14 13:24:44,502140,(10688) BD237,00009cecf5dd879db875634471b58705ade02176dde186...,(12758) BD237_V2,(02) Urbano,20260414,13:24:44,...,12758,BD237_V2,13,2,79,4.683434,-74.090934,2026-04-16 13:24:44,370A05,2026-04-16 13:51:29
1776960,(53008) 370A05_TM|370A05_Br. Nuevo Milenio,2026-04-14,2026-04-14 13:51:29,504389,(10311) DA213,00009cecf5dd879db875634471b58705ade02176dde186...,(12730) DA213_V2,(02) Urbano,20260414,13:51:29,...,12730,DA213_V2,13,2,106,4.711427,-74.136280,2026-04-16 13:51:29,NaN,NaT


In [253]:
map_orden = od.drop_duplicates(subset=['Ruta_comercial', 'Parada']) \
    .set_index(['Ruta_comercial', 'Parada'])['orden_parada']

In [254]:
od['orden_destino'] = od.set_index(['Ruta_comercial', 'Parada_siguiente']).index.map(map_orden)

od.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,...,Sentido,Franja,id_tarjeta,orden_parada,latitud,longitud,Hora_dt,Parada_siguiente,Hora_siguiente,orden_destino
1734142,(52692) 181C05_TM|181C05_Br. La Estrada,2026-04-14,2026-04-14 08:37:14,502121,(10688) BD237,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,(12758) BD237_V2,(02) Urbano,20260414,08:37:14,...,BD237_V2,8,0,26,4.683280,-74.091076,2026-04-16 08:37:14,NaN,NaT,NaN
1729151,(57003) 103B00_TM|103B00_Br. Quesada,2026-04-14,2026-04-14 08:09:53,502080,(10244) SE10,0000652dcb91a8643aa46f9384dfec0d41bd92a9470e2f...,(12741) SE10_V3,(02) Urbano,20260414,08:09:53,...,SE10_V3,8,1,71,4.638265,-74.069733,2026-04-16 08:09:53,NaN,NaT,NaN
1691500,(52370) 070B05_TM|070B05_Br. Sabana del Dorado,2026-04-14,2026-04-14 05:44:54,507061,(10339) C25,00009cecf5dd879db875634471b58705ade02176dde186...,(12756) C25_V3,(02) Urbano,20260414,05:44:54,...,C25_V3,5,2,5,4.709103,-74.142388,2026-04-16 05:44:54,157B04,2026-04-16 13:24:44,NaN
1773353,(51465) 157B04_TM|157B04_Br. Las Ferias,2026-04-14,2026-04-14 13:24:44,502140,(10688) BD237,00009cecf5dd879db875634471b58705ade02176dde186...,(12758) BD237_V2,(02) Urbano,20260414,13:24:44,...,BD237_V2,13,2,79,4.683434,-74.090934,2026-04-16 13:24:44,370A05,2026-04-16 13:51:29,NaN
1776960,(53008) 370A05_TM|370A05_Br. Nuevo Milenio,2026-04-14,2026-04-14 13:51:29,504389,(10311) DA213,00009cecf5dd879db875634471b58705ade02176dde186...,(12730) DA213_V2,(02) Urbano,20260414,13:51:29,...,DA213_V2,13,2,106,4.711427,-74.136280,2026-04-16 13:51:29,NaN,NaT,NaN


In [255]:
od_final = od[[
    "id_tarjeta",
    "Parada",
    "orden_parada",
    "Hora_dt",
    "Parada_siguiente",
    "orden_destino",
    "Hora_siguiente",
    "Ruta_comercial",
    "ID_Vehiculo",
    "Sentido",
    "Franja",
    "latitud",
    "longitud"
]].rename(columns={
    "Parada": "origen_parada",
    "Parada_siguiente": "destino_parada",
    "Hora_dt": "origen_hora",
    "Hora_siguiente": "destino_hora"
})

od_final.head()

,id_tarjeta,origen_parada,orden_parada,origen_hora,destino_parada,orden_destino,destino_hora,Ruta_comercial,ID_Vehiculo,Sentido,Franja,latitud,longitud
1734142,0,181C05,26,2026-04-16 08:37:14,NaN,NaN,NaT,BD237,502121,BD237_V2,8,4.683280,-74.091076
1729151,1,103B00,71,2026-04-16 08:09:53,NaN,NaN,NaT,SE10,502080,SE10_V3,8,4.638265,-74.069733
1691500,2,070B05,5,2026-04-16 05:44:54,157B04,NaN,2026-04-16 13:24:44,C25,507061,C25_V3,5,4.709103,-74.142388
1773353,2,157B04,79,2026-04-16 13:24:44,370A05,NaN,2026-04-16 13:51:29,BD237,502140,BD237_V2,13,4.683434,-74.090934
1776960,2,370A05,106,2026-04-16 13:51:29,NaN,NaN,NaT,DA213,504389,DA213_V2,13,4.711427,-74.136280


In [256]:
od_finalp = od_final.sort_values([
    'Ruta_comercial',
    'Sentido',
    'ID_Vehiculo',
    'Franja',
    'origen_hora'
])

od_finalp.head()

,id_tarjeta,origen_parada,orden_parada,origen_hora,destino_parada,orden_destino,destino_hora,Ruta_comercial,ID_Vehiculo,Sentido,Franja,latitud,longitud
1690302,41537,408A06,3,2026-04-16 05:39:35,NaN,NaN,NaT,12,502003,12,5,4.685737,-74.157909
1691131,43535,504A06,7,2026-04-16 05:43:22,124A06,191.0,2026-04-16 15:56:26,12,502003,12,5,NaN,NaN
1691479,59247,441A06,9,2026-04-16 05:44:47,480A05,143.0,2026-04-16 17:39:13,12,502003,12,5,4.687194,-74.155649
1691492,121916,441A06,9,2026-04-16 05:44:51,125A05,127.0,2026-04-16 12:45:05,12,502003,12,5,4.687194,-74.155649
1691506,36683,441A06,9,2026-04-16 05:44:56,NaN,NaN,NaT,12,502003,12,5,4.687194,-74.155649


In [257]:
od_finalp['usuario_desciende'] = od_finalp['destino_parada'].notna().astype(int)

od_finalp.head()

,id_tarjeta,origen_parada,orden_parada,origen_hora,destino_parada,orden_destino,destino_hora,Ruta_comercial,ID_Vehiculo,Sentido,Franja,latitud,longitud,usuario_desciende
1690302,41537,408A06,3,2026-04-16 05:39:35,NaN,NaN,NaT,12,502003,12,5,4.685737,-74.157909,0
1691131,43535,504A06,7,2026-04-16 05:43:22,124A06,191.0,2026-04-16 15:56:26,12,502003,12,5,NaN,NaN,1
1691479,59247,441A06,9,2026-04-16 05:44:47,480A05,143.0,2026-04-16 17:39:13,12,502003,12,5,4.687194,-74.155649,1
1691492,121916,441A06,9,2026-04-16 05:44:51,125A05,127.0,2026-04-16 12:45:05,12,502003,12,5,4.687194,-74.155649,1
1691506,36683,441A06,9,2026-04-16 05:44:56,NaN,NaN,NaT,12,502003,12,5,4.687194,-74.155649,0


In [258]:
od_finalp['sube'] = 1
od_finalp['baja'] = od_finalp['usuario_desciende']

od_finalp.head()

,id_tarjeta,origen_parada,orden_parada,origen_hora,destino_parada,orden_destino,destino_hora,Ruta_comercial,ID_Vehiculo,Sentido,Franja,latitud,longitud,usuario_desciende,sube,baja
1690302,41537,408A06,3,2026-04-16 05:39:35,NaN,NaN,NaT,12,502003,12,5,4.685737,-74.157909,0,1,0
1691131,43535,504A06,7,2026-04-16 05:43:22,124A06,191.0,2026-04-16 15:56:26,12,502003,12,5,NaN,NaN,1,1,1
1691479,59247,441A06,9,2026-04-16 05:44:47,480A05,143.0,2026-04-16 17:39:13,12,502003,12,5,4.687194,-74.155649,1,1,1
1691492,121916,441A06,9,2026-04-16 05:44:51,125A05,127.0,2026-04-16 12:45:05,12,502003,12,5,4.687194,-74.155649,1,1,1
1691506,36683,441A06,9,2026-04-16 05:44:56,NaN,NaN,NaT,12,502003,12,5,4.687194,-74.155649,0,1,0


In [259]:
od_finalp['conteo_pasajeros'] = od_finalp.groupby(
    ['Ruta_comercial', 'Sentido', 'ID_Vehiculo', 'Franja']
).apply(
    lambda x: (x['sube'] - x['baja']).cumsum()
).reset_index(level=[0,1,2,3], drop=True)

od_finalp.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_28816\2394391313.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(


,id_tarjeta,origen_parada,orden_parada,origen_hora,destino_parada,orden_destino,destino_hora,Ruta_comercial,ID_Vehiculo,Sentido,Franja,latitud,longitud,usuario_desciende,sube,baja,conteo_pasajeros
1690302,41537,408A06,3,2026-04-16 05:39:35,NaN,NaN,NaT,12,502003,12,5,4.685737,-74.157909,0,1,0,1
1691131,43535,504A06,7,2026-04-16 05:43:22,124A06,191.0,2026-04-16 15:56:26,12,502003,12,5,NaN,NaN,1,1,1,1
1691479,59247,441A06,9,2026-04-16 05:44:47,480A05,143.0,2026-04-16 17:39:13,12,502003,12,5,4.687194,-74.155649,1,1,1,1
1691492,121916,441A06,9,2026-04-16 05:44:51,125A05,127.0,2026-04-16 12:45:05,12,502003,12,5,4.687194,-74.155649,1,1,1,1
1691506,36683,441A06,9,2026-04-16 05:44:56,NaN,NaN,NaT,12,502003,12,5,4.687194,-74.155649,0,1,0,2


In [260]:
validaciones.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Validaciones/{fecha}_validaciones_procesadas.csv', index= False, sep=';')

In [261]:
od.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Validaciones/{fecha}_origen_destino.csv', index= False, sep=';')

In [262]:
df_tarjetas.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Validaciones/{fecha}_transaciones_tarjeta.csv', index= False, sep=';')

In [263]:
od_final.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Validaciones/{fecha}_origen_destino_paradas.csv', index= False, sep=';')

In [264]:
od_finalp.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Validaciones/{fecha}_origen_destino_pax.csv', index= False, sep=';')